# Early-Fusion Text-Serialized Transformer
### Binary Classification: Multi-hop (`0`) vs Single-hop Shortcut (`1`)

**Architecture:** ModernBERT-base (`answerdotai/ModernBERT-base`) + LoRA (PEFT)  
**Device:** Apple Silicon MPS / CPU  
**Pipeline:** HuggingFace `Trainer` with cosine LR scheduler, best checkpoint tracking, and diagnostic evaluation.

## Section 1 – Imports & Device Detection

In [ ]:
# Optional: install packages if needed
# !pip install transformers datasets peft scikit-learn seaborn matplotlib pandas numpy accelerate

In [ ]:
import os, sys, warnings, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
)
from sklearn.model_selection import train_test_split

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")
set_seed(42)

# ── Device Setup (Apple Silicon MPS / CUDA / CPU) ─────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Device: Apple Silicon MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Device: CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Device: CPU (training will be slower)")

print(f"PyTorch version    : {torch.__version__}")
print(f"Transformers version: {sys.modules.get('transformers').__version__ if 'transformers' in sys.modules else 'installed'}")

## Section 2 – Configuration & Path Setup

In [ ]:
# ── Robust Project Root Resolution ────────────────────────────────────
# Automatically locates project root regardless of notebook directory
current_dir = Path.cwd().resolve()
for candidate in [current_dir, current_dir.parent, current_dir.parents[1]]:
    if (candidate / "data" / "processed").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = Path("/Users/anni/Desktop/SOURABH/research project/Multihop-Shortcut-Detector-Rag")

DATA_PATH  = REPO_ROOT / "data" / "processed" / "classifier_dataset.jsonl"
OUTPUT_DIR = REPO_ROOT / "models" / "early_fusion"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)

# ── Model & LoRA Parameters ───────────────────────────────────────────
MODEL_NAME = "answerdotai/ModernBERT-base"   # fallback: "microsoft/deberta-v3-base"
NUM_LABELS = 2

LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.1

# Module target selection
if "modernbert" in MODEL_NAME.lower():
    LORA_TARGET_MODS = ["Wqkv", "Wo"]
elif "deberta" in MODEL_NAME.lower():
    LORA_TARGET_MODS = ["query_proj", "key_proj", "value_proj", "dense"]
else:
    LORA_TARGET_MODS = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training Hyperparameters ──────────────────────────────────────────
MAX_LENGTH        = 256
BATCH_TRAIN       = 16
BATCH_EVAL        = 32
NUM_EPOCHS        = 5
LR                = 5e-5
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1

print("Configuration ready:")
print(f"  Repo root     : {REPO_ROOT}")
print(f"  Dataset path  : {DATA_PATH}")
print(f"  Output dir    : {OUTPUT_DIR}")
print(f"  Model name    : {MODEL_NAME}")
print(f"  LoRA modules  : {LORA_TARGET_MODS}")
print(f"  Batch / Epochs: {BATCH_TRAIN} / {NUM_EPOCHS}")

## Section 3 – Data Loading & Synchronized Stratified Splits

In [ ]:
# Load full dataset from JSONL
print(f"Reading dataset: {DATA_PATH}")
raw_df = pd.read_json(DATA_PATH, lines=True)

# 1. Filter out discarded examples (label == -1) and exact duplicate rows
clean_df = raw_df[raw_df["label"] != -1].drop_duplicates().reset_index(drop=True)

# 2. Stratified 80/20 train+val vs test split (random_state=42)
train_val_df, test_df = train_test_split(
    clean_df, test_size=0.20, random_state=42, stratify=clean_df["label"]
)

# 3. Stratified 70/10 train vs val split from the 80% train+val set (0.125 * 0.8 = 0.10)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.125, random_state=42, stratify=train_val_df["label"]
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Identify numerical features (everything except id, question, label)
FEATURE_COLS = [
    c for c in train_df.columns if c not in ["id", "question", "label"]
]

print("✅ Synchronized splits created without index mismatch:")
print(f"  train_df : {train_df.shape} -> (Multi-hop: {(train_df.label==0).sum()}, Shortcut: {(train_df.label==1).sum()})")
print(f"  val_df   : {val_df.shape} -> (Multi-hop: {(val_df.label==0).sum()}, Shortcut: {(val_df.label==1).sum()})")
print(f"  test_df  : {test_df.shape} -> (Multi-hop: {(test_df.label==0).sum()}, Shortcut: {(test_df.label==1).sum()})")
print(f"  Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
train_df.head(2)

## Section 4 – Text Serialization Function

Converts the numerical retrieval features into a compact, diagnostic prefix and prepends it to the question string:
`"[RET] RRF_Top1: 0.84 | Margin: 0.21 | Entropy: 0.14 | Overlap: 1.00 | All_Ent: 1 | Comp: 0 [QUERY] {question}"`

In [ ]:
# Short readable names for token-efficient diagnostic prefix
FEAT_LABELS = {
    "rrf_score_top1":        "RRF_Top1",
    "rrf_margin":            "Margin",
    "rrf_score_entropy":     "Entropy",
    "entity_overlap_ratio":  "Overlap",
    "has_all_query_entities":"All_Ent",
    "query_token_count":     "QTok",
    "is_comparison":         "Comp",
}

def serialize_row(row: pd.Series, feature_cols: list) -> str:
    """Converts row retrieval features + question into a structured serialized string."""
    parts = []
    for col in feature_cols:
        short_name = FEAT_LABELS.get(col, col)
        val = row[col]
        if isinstance(val, (float, np.floating)):
            parts.append(f"{short_name}: {val:.2f}")
        else:
            parts.append(f"{short_name}: {int(val)}")
    ret_prefix = " | ".join(parts)
    return f"[RET] {ret_prefix} [QUERY] {row['question']}"

# Apply serialization across all three splits
train_df["text_input"] = train_df.apply(serialize_row, axis=1, feature_cols=FEATURE_COLS)
val_df["text_input"]   = val_df.apply(serialize_row, axis=1, feature_cols=FEATURE_COLS)
test_df["text_input"]  = test_df.apply(serialize_row, axis=1, feature_cols=FEATURE_COLS)

print("── Example Serialized Input ──")
print(train_df["text_input"].iloc[0])
print()
print(f"Estimated character count: {train_df['text_input'].str.len().mean():.0f} (~{train_df['text_input'].str.len().mean()/4:.0f} tokens)")

## Section 5 – Tokenizer & Dataset Preparation

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Register [RET] and [QUERY] special tokens
special_tokens = {"additional_special_tokens": ["[RET]", "[QUERY]"]}
tokenizer.add_special_tokens(special_tokens)

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    """Converts a DataFrame to a tokenized HuggingFace Dataset."""
    ds = Dataset.from_pandas(df[["text_input", "label"]].reset_index(drop=True))
    def tokenize_fn(batch):
        return tokenizer(
            batch["text_input"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,   # dynamic padding via DataCollator
        )
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text_input"])
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")
    return ds

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

print(f"train_ds: {train_ds}")
print(f"val_ds  : {val_ds}")
print(f"test_ds : {test_ds}")
print(f"Token length for sample 0: {len(train_ds[0]['input_ids'])} tokens (max_length={MAX_LENGTH})")

## Section 6 – Model & PEFT LoRA Setup

In [ ]:
# Load base sequence classification model
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)

# Resize token embeddings to include [RET] and [QUERY] tokens
base_model.resize_token_embeddings(len(tokenizer))

# Human-readable labels
base_model.config.id2label = {0: "Multi-hop", 1: "Single-hop"}
base_model.config.label2id = {"Multi-hop": 0, "Single-hop": 1}

# Configure PEFT LoRA
lora_config = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGET_MODS,
    bias           = "none",
    inference_mode = False,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model = model.to(DEVICE)

## Section 7 – Evaluation Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    """Computes Accuracy, Macro F1, and ROC-AUC per evaluation step."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Numerical softmax
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    pos_probs = probs[:, 1]

    acc      = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    roc_auc  = roc_auc_score(labels, pos_probs)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "roc_auc":  roc_auc,
    }

## Section 8 – Trainer Configuration

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR / "checkpoints"),
    overwrite_output_dir        = True,

    # Checkpoint & evaluation strategy
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "macro_f1",
    greater_is_better           = True,
    save_total_limit            = 2,

    # Optimization
    learning_rate               = LR,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    num_train_epochs            = NUM_EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",

    # Apple Silicon MPS / CPU settings
    fp16                        = False,
    bf16                        = False,
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,

    # Logging
    logging_dir                 = str(OUTPUT_DIR / "logs"),
    logging_strategy            = "steps",
    logging_steps               = 25,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print("Trainer initialized successfully:")
print(f"  Epochs           : {NUM_EPOCHS}")
print(f"  Learning Rate    : {LR}")
print(f"  Train Batch Size : {BATCH_TRAIN}")
print(f"  LR Scheduler     : cosine with {WARMUP_RATIO*100:.0f}% warmup")
print(f"  Best Metric      : macro_f1")

## Section 9 – Training

In [ ]:
train_result = trainer.train()

# Save final best LoRA adapter & tokenizer
final_model_path = OUTPUT_DIR / "final"
trainer.save_model(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

print("\n══════════════════════════════════════════════════════")
print("              TRAINING COMPLETE                       ")
print("══════════════════════════════════════════════════════")
print(f"  Total Steps    : {train_result.global_step}")
print(f"  Train Loss     : {train_result.training_loss:.4f}")
print(f"  Train Time (s) : {train_result.metrics['train_runtime']:.1f}")
print(f"  Saved Model to : {final_model_path}")

## Section 10 – Evaluation on Test Set (`test_df`: 446 rows)

In [ ]:
# Predict on test set
pred_output = trainer.predict(test_ds)
logits      = pred_output.predictions
test_labels = pred_output.label_ids
test_preds  = np.argmax(logits, axis=-1)

# Softmax for positive class probabilities
exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
test_probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
pos_probs  = test_probs[:, 1]

# Overall metrics
test_acc      = accuracy_score(test_labels, test_preds)
test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
test_roc_auc  = roc_auc_score(test_labels, pos_probs)

print("══════════════════════════════════════════════════════")
print("           TEST SET EVALUATION RESULTS                ")
print("══════════════════════════════════════════════════════")
print(f"  Accuracy : {test_acc:.4f}")
print(f"  Macro F1 : {test_macro_f1:.4f}")
print(f"  ROC-AUC  : {test_roc_auc:.4f}")
print("──────────────────────────────────────────────────────")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=["Multi-hop (0)", "Single-hop (1)"]))

## Section 11 – Confusion Matrix (Highlighting False Positives)

**Critical Safety Signal:** False Positives occur when an actual **Multi-hop** question (`0`) is misclassified as a **Shortcut** (`1`), meaning the pipeline fails to retrieve sufficient hops.

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
class_names = ["Multi-hop (0)", "Single-hop (1)"]

fig, ax = plt.subplots(figsize=(6.5, 5))

# Plot heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=1.5,
    linecolor="white",
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 16, "weight": "bold"},
    ax=ax,
)

# False Positives cell is at Row 0, Column 1 (x: 1..2, y: 0..1 in heatmap coordinates)
fp_count = cm[0, 1]
ax.add_patch(
    plt.Rectangle(
        (1, 0), 1, 1,
        fill=False,
        edgecolor="#FF3D00",
        lw=4.0,
        transform=ax.transData
    )
)

# Text highlight in FP cell
ax.text(
    1.5, 0.25,
    f"⚠️ False Positives: {fp_count}",
    ha="center", va="center",
    color="#D50000",
    fontsize=11,
    fontweight="bold",
    transform=ax.transData,
)

ax.set_title("Confusion Matrix — Early-Fusion Transformer\nHighlighted Border: False Positives (Multi-hop mistaken for Shortcut)", fontsize=11, pad=12)
ax.set_xlabel("Predicted Label", fontsize=12, labelpad=8)
ax.set_ylabel("True Label", fontsize=12, labelpad=8)

plt.tight_layout()
cm_plot_path = OUTPUT_DIR / "plots" / "confusion_matrix_test.png"
plt.savefig(cm_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved to: {cm_plot_path}")

## Section 12 – Training Loss and Metric Curves

In [ ]:
history = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Loss Curve
train_loss = history.dropna(subset=["loss"])
eval_loss  = history.dropna(subset=["eval_loss"])

if not train_loss.empty:
    axes[0].plot(train_loss["step"], train_loss["loss"], label="Train Loss", color="#1976D2", lw=2)
if not eval_loss.empty:
    axes[0].plot(eval_loss["step"], eval_loss["eval_loss"], "o-", label="Val Loss", color="#D32F2F", lw=2)
axes[0].set_title("Cross-Entropy Loss")
axes[0].set_xlabel("Step")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Macro F1 Curve
if "eval_macro_f1" in history.columns:
    eval_f1 = history.dropna(subset=["eval_macro_f1"])
    axes[1].plot(eval_f1["epoch"], eval_f1["eval_macro_f1"], "o-", color="#388E3C", lw=2)
    axes[1].set_title("Validation Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(True, alpha=0.3)

# 3. ROC-AUC Curve
if "eval_roc_auc" in history.columns:
    eval_auc = history.dropna(subset=["eval_roc_auc"])
    axes[2].plot(eval_auc["epoch"], eval_auc["eval_roc_auc"], "o-", color="#7B1FA2", lw=2)
    axes[2].set_title("Validation ROC-AUC")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = OUTPUT_DIR / "plots" / "training_curves.png"
plt.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Curves saved to: {curves_path}")

## Section 13 – Production Inference Function

In [ ]:
def predict_shortcut(question: str, retrieval_features: dict) -> dict:
    """Predicts whether a question needs Multi-hop (0) or is a Shortcut (1).

    Args:
        question: raw question string
        retrieval_features: dictionary of numerical features matching FEATURE_COLS

    Returns:
        dict with label (0 or 1), label_name, confidence, and class probabilities
    """
    # Create single-row series for serialization
    row = pd.Series({**retrieval_features, "question": question})
    text_input = serialize_row(row, FEATURE_COLS)

    # Tokenize
    inputs = tokenizer(
        text_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_class = int(np.argmax(probs))
    label_map  = {0: "Multi-hop Required (0)", 1: "Single-hop Shortcut (1)"}

    return {
        "text_input" : text_input,
        "label"      : pred_class,
        "label_name" : label_map[pred_class],
        "confidence" : float(probs[pred_class]),
        "prob_multihop": float(probs[0]),
        "prob_shortcut": float(probs[1]),
    }

# Demo inference
demo_question = "Which magazine was started first Arthur's Magazine or First for Women?"
demo_features = {
    "rrf_score_top1"        : 0.0325,
    "rrf_margin"            : 0.0003,
    "rrf_score_entropy"     : 4.52,
    "entity_overlap_ratio"  : 0.67,
    "has_all_query_entities": 0,
    "query_token_count"     : 11,
    "is_comparison"         : 1,
}

result = predict_shortcut(demo_question, demo_features)
print("Demo Input Question :", demo_question)
print("Serialized Text     :", result["text_input"])
print("Prediction Result   :", result["label_name"])
print(f"Confidence          : {result['confidence']*100:.1f}%")